# Assignment 3: Two Averages, One Argument (statistics)

Source: `teaching_prices_clean.csv`, 96 rows, 6 columns, all prices numeric.
SHA-256: `a298f754d72600bdf0552404ebe7a37530c8ca9dd8fc7178945f00f915eb45cd`

**Notebook map**

1. Integrity checks on the cleaned file.
2. Mean, median and first-to-last change for each item.
3. Is the first-to-last change representative? Monthly averages and weekly direction.
4. Week 2 sensitivity: mean and median with and without a flagged value.
5. Pooled versus item-level averages, and the 6.0 PKR city gap.
6. Summary and assertions.

The draft referee's note, with O/E/C labels, is in `1_attempt.md`. The AI interpretation audit and the classmate excerpt are in `3_log_reflection.md`.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)

DATA_PATH = Path("teaching_prices_clean.csv")
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

print("file:", DATA_PATH)
print("sha256:", DATA_SHA256)
print("pandas:", pd.__version__)

file: teaching_prices_clean.csv
sha256: a298f754d72600bdf0552404ebe7a37530c8ca9dd8fc7178945f00f915eb45cd
pandas: 3.0.2


## 1. Integrity checks

Before any statistic, check that the file is the balanced panel it claims to be. The Week 3 file has no missing prices. The Week 2 file is used later only for the flagged-value sensitivity, because it deliberately keeps its three invalid prices missing.

In [2]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"])

print("shape:", df.shape)
print()
print(df.dtypes.to_string())
df.head(8)

shape: (96, 6)

date          datetime64[us]
item                     str
city                     str
price                float64
unit                     str
data_label               str


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
5,2026-01-01,Pulse C,City 2,206.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
6,2026-01-01,Sweetener D,City 1,149.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
7,2026-01-01,Sweetener D,City 2,155.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [3]:
integrity = pd.Series({
    "rows": len(df),
    "columns": df.shape[1],
    "missing cells": int(df.isna().sum().sum()),
    "duplicate item-city-date keys": int(df.duplicated(["date", "item", "city"]).sum()),
    "items": df["item"].nunique(),
    "cities": df["city"].nunique(),
    "dates": df["date"].nunique(),
    "rows per item (min)": int(df.groupby("item").size().min()),
    "rows per item (max)": int(df.groupby("item").size().max()),
    "rows per item-city (min)": int(df.groupby(["item", "city"]).size().min()),
    "rows per date (min)": int(df.groupby("date").size().min()),
    "price min": df["price"].min(),
    "price max": df["price"].max(),
}, dtype="object")
integrity

rows                                96
columns                              6
missing cells                        0
duplicate item-city-date keys        0
items                                4
cities                               2
dates                               12
rows per item (min)                 24
rows per item (max)                 24
rows per item-city (min)            12
rows per date (min)                  8
price min                         99.5
price max                        350.0
dtype: object

### Data dictionary

In [4]:
pd.DataFrame([
    ("date", "datetime64", "Weekly observation date, every Thursday, 2026-01-01 to 2026-03-19"),
    ("item", "category", "4 items: Oil B, Pulse C, Staple A, Sweetener D"),
    ("city", "category", "City 1, City 2"),
    ("price", "float", "PKR per kg (illustrative); 96 numeric values in this file"),
    ("unit", "text (constant)", "PKR per kg (illustrative)"),
    ("data_label", "text (constant)", "ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS"),
], columns=["column", "type", "meaning"])

,column,type,meaning
0,date,datetime64,"Weekly observation date, every Thursday, 2026-..."
1,item,category,"4 items: Oil B, Pulse C, Staple A, Sweetener D"
2,city,category,"City 1, City 2"
3,price,float,PKR per kg (illustrative); 96 numeric values i...
4,unit,text (constant),PKR per kg (illustrative)
5,data_label,text (constant),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 2. Mean, median and first-to-last change per item

The mean and median use all 24 observations for each item (12 weeks × 2 cities). First-to-last is computed on the **item average across the two cities at each date**, so the two cities are weighted equally and the comparison is not a City 1 first week against a City 2 last week. Per-city changes are shown in the next table as a cross-check.

In [5]:
item_date = df.groupby(["item", "date"], as_index=False)["price"].mean()

first_last = item_date.sort_values("date").groupby("item").agg(
    first=("price", "first"),
    last=("price", "last"),
)

stats = df.groupby("item")["price"].agg(
    n="count",
    mean="mean",
    median="median",
    min="min",
    max="max",
).join(first_last)

stats["mean_minus_median"] = stats["mean"] - stats["median"]
stats["pct_change_first_last"] = (stats["last"] / stats["first"] - 1) * 100
stats.round(3)

,n,mean,median,min,max,first,last,mean_minus_median,pct_change_first_last
item,,,,,,,,,
Oil B,24,324.833,324.00,298.0,350.0,301.0,347.0,0.833,15.282
Pulse C,24,205.917,206.00,200.0,213.5,203.0,210.5,-0.083,3.695
Staple A,24,111.167,111.75,99.5,122.0,104.0,117.5,-0.583,12.981
Sweetener D,24,166.750,166.75,149.0,184.5,152.0,181.5,0.000,19.408


In [6]:
def item_city_table(frame):
    out = frame.groupby(["item", "city"])["price"].agg(
        n="count",
        mean="mean",
        median="median",
    )
    fl = frame.sort_values("date").groupby(["item", "city"]).agg(
        first=("price", "first"),
        last=("price", "last"),
    )
    out = out.join(fl)
    out["pct_change_first_last"] = (out["last"] / out["first"] - 1) * 100
    return out.round(3)

item_city = item_city_table(df)
item_city

n     mean  median  first   last  pct_change_first_last
item        city                                                            
Oil B       City 1  12  321.833  321.00  298.0  344.0                 15.436
            City 2  12  327.833  327.00  304.0  350.0                 15.132
Pulse C     City 1  12  202.917  202.50  200.0  207.5                  3.750
            City 2  12  208.917  208.50  206.0  213.5                  3.641
Staple A    City 1  12  108.167  107.75  101.0  114.5                 13.366
            City 2  12  114.167  113.75  107.0  120.5                 12.617
Sweetener D City 1  12  163.750  163.75  149.0  178.5                 19.799
            City 2  12  169.750  169.75  155.0  184.5                 19.032

### Which statistic I would report, and why

| Item | Level to report | Change to report | Reason |
|---|---|---|---|
| **Staple A** | **Median 111.75** | **+12.98%** | The mean is 111.17, only 0.58 lower. The median is the safer “typical price” label because it is not moved by one extreme value. The change is confirmed by the monthly averages 106.00 → 113.00 → 117.33. |
| **Oil B** | **Mean 324.83** | **+15.28%** | The median is 324.00, a difference of 0.83. The series rises in all 11 weekly steps in both cities, so the mean is not distorted by a few points and the endpoint change is representative. |
| **Pulse C** | **Median 206.00** | **+3.69%**, described as a small rise | The mean is 205.92, a difference of 0.08. There are four down weeks in each city and only a 3.69% endpoint change, so the median and monthly averages are safer than a headline percentage. |
| **Sweetener D** | **Either; mean = median = 166.75** | **+19.41%** | The two summaries are identical, and the rise is confirmed by the monthly averages 158.00 → 168.75 → 178.67. |

In this clean file the choice between mean and median changes the level by at most 0.83 PKR per item, so the choice does not decide the direction. The choice matters when a flagged value is present; that is the Week 2 check in §4.

## 3. Is the first-to-last change representative?

A first-to-last change uses only two weeks. Check it against the monthly averages and the weekly path. If the series is monotonic, the endpoint measure is safe. If it wobbles, describe the number as an endpoint comparison, not as a smooth trend.

In [7]:
df["month"] = df["date"].dt.strftime("%Y-%m")
monthly = df.groupby(["item", "month"])["price"].mean().unstack("month")
monthly["Jan->Mar %"] = (monthly["2026-03"] / monthly["2026-01"] - 1) * 100
monthly.round(2)

month,2026-01,2026-02,2026-03,Jan->Mar %
item,,,,
Oil B,311.0,328.75,342.67,10.18
Pulse C,204.0,206.75,208.00,1.96
Staple A,106.0,113.00,117.33,10.69
Sweetener D,158.0,168.75,178.67,13.08


In [8]:
def weekly_direction(frame):
    rows = []
    for (item, city), grp in frame.sort_values("date").groupby(["item", "city"]):
        diff = grp["price"].diff().dropna()
        rows.append({
            "item": item,
            "city": city,
            "up weeks": int((diff > 0).sum()),
            "down weeks": int((diff < 0).sum()),
            "flat weeks": int((diff == 0).sum()),
            "largest weekly fall": diff.min(),
        })
    return pd.DataFrame(rows)

direction = weekly_direction(df)
direction

,item,city,up weeks,down weeks,flat weeks,largest weekly fall
0,Oil B,City 1,11,0,0,1.0
1,Oil B,City 2,11,0,0,1.0
2,Pulse C,City 1,7,4,0,-2.5
3,Pulse C,City 2,7,4,0,-2.5
4,Staple A,City 1,6,5,0,-1.5
5,Staple A,City 2,6,5,0,-1.5
6,Sweetener D,City 1,7,4,0,-0.5
7,Sweetener D,City 2,7,4,0,-0.5


In [9]:
weekly = item_date.pivot(index="date", columns="item", values="price")
weekly.round(2)

item,Oil B,Pulse C,Staple A,Sweetener D
date,,,,
2026-01-01,301.0,203.0,104.0,152.0
2026-01-08,307.0,205.5,102.5,156.5
2026-01-15,313.0,203.0,106.0,156.0
2026-01-22,314.0,205.5,109.5,160.5
2026-01-29,320.0,203.0,108.0,165.0
2026-02-05,321.0,205.5,111.5,164.5
2026-02-12,327.0,208.0,110.0,169.0
2026-02-19,333.0,205.5,113.5,168.5
2026-02-26,334.0,208.0,117.0,173.0


Oil B rises in every week in both cities, so its endpoint change is a clean summary. Pulse C, Staple A and Sweetener D each have four or five down weeks; their first-to-last numbers are still positive because the last week is above the first, but they are endpoint comparisons rather than smooth trends. The monthly averages rise for all four items, which is why the draft note uses the monthly check before treating the direction as robust.

## 4. Week 2 flagged-value sensitivity

The Week 2 clean file has 23 numeric Staple A values, 23 Sweetener D values and 23 Pulse C values. Each item is missing one price: Pulse C is blank in the source, Staple A carries the flagged `1,250`, and Sweetener D carries the flagged `9,999`. Putting a flagged value back into its row gives the “with flag” comparison required by the brief.

In [10]:
W2_PATH = Path("../week2/teaching_prices_clean.csv")
w2 = pd.read_csv(W2_PATH, parse_dates=["date"])

print("Week 2 clean file:", W2_PATH)
print("shape:", w2.shape)
print()
print("missing prices by item:")
print(w2[w2["price"].isna()].groupby("item").size().to_string())
print()
w2[w2["price"].isna()][["date", "item", "city", "price"]]

Week 2 clean file: ../week2/teaching_prices_clean.csv
shape: (96, 6)

missing prices by item:
item
Pulse C        1
Staple A       1
Sweetener D    1



,date,item,city,price
3,2026-01-01,Pulse C,City 2,NaN
21,2026-01-15,Staple A,City 2,NaN
30,2026-01-22,Sweetener D,City 1,NaN


In [11]:
FLAGGED = {
    "Staple A": 1250.0,    # Week 2 row_id 17, flagged in the Week 2 audit
    "Sweetener D": 9999.0, # Week 2 row_id 30, flagged in the Week 2 audit
}

def flag_sensitivity(frame, item, flag):
    clean = frame.loc[(frame["item"] == item) & frame["price"].notna(), "price"]
    with_flag = pd.concat([clean, pd.Series([flag])], ignore_index=True)
    return pd.Series({
        "n clean": len(clean),
        "mean clean": clean.mean(),
        "median clean": clean.median(),
        "flagged value": flag,
        "n with flag": len(with_flag),
        "mean with flag": with_flag.mean(),
        "median with flag": with_flag.median(),
        "mean change %": (with_flag.mean() / clean.mean() - 1) * 100,
        "median change %": (with_flag.median() / clean.median() - 1) * 100,
    })

sensitivity = pd.DataFrame({
    item: flag_sensitivity(w2, item, flag) for item, flag in FLAGGED.items()
}).T

sensitivity.round(3)

,n clean,mean clean,median clean,flagged value,n with flag,mean with flag,median with flag,mean change %,median change %
Staple A,23.0,111.261,112.5,1250.0,24.0,158.708,112.50,42.645,0.000
Sweetener D,23.0,167.152,167.5,9999.0,24.0,576.812,167.75,245.082,0.149


In [12]:
# The mean shift is the extra total divided across all observations.
rows = []
for item, flag in FLAGGED.items():
    clean = w2.loc[(w2["item"] == item) & w2["price"].notna(), "price"]
    n_with = len(clean) + 1
    rows.append({
        "item": item,
        "clean mean": clean.mean(),
        "flagged value": flag,
        "extra total": flag - clean.mean(),
        "n with flag": n_with,
        "added to mean": (flag - clean.mean()) / n_with,
        "mean with flag": clean.mean() + (flag - clean.mean()) / n_with,
    })

pd.DataFrame(rows).round(3)

,item,clean mean,flagged value,extra total,n with flag,added to mean,mean with flag
0,Staple A,111.261,1250.0,1138.739,24,47.447,158.708
1,Sweetener D,167.152,9999.0,9831.848,24,409.660,576.812


In [13]:
# The median is positional: show the middle values with and without the flag.
rows = []
for item, flag in FLAGGED.items():
    clean = w2.loc[(w2["item"] == item) & w2["price"].notna(), "price"].sort_values()
    with_flag = pd.concat([clean, pd.Series([flag])]).sort_values().reset_index(drop=True)
    rows.append({
        "item": item,
        "middle of 23 (12th)": clean.iloc[11],
        "middle of 24 (12th, 13th)": f"{with_flag.iloc[11]:.1f}, {with_flag.iloc[12]:.1f}",
        "median clean": clean.median(),
        "median with flag": with_flag.median(),
    })

pd.DataFrame(rows)

,item,middle of 23 (12th),"middle of 24 (12th, 13th)",median clean,median with flag
0,Staple A,112.5,"112.5, 112.5",112.5,112.50
1,Sweetener D,167.5,"167.5, 168.0",167.5,167.75


The 1,250 adds 1,138.74 to Staple A’s total, which is 47.45 per observation across 24, so the mean rises from 111.26 to 158.71. The median is the average of the 12th and 13th values; the flag is the largest value, so the middle pair stays 112.5 and 112.5. The 9,999 case is the same arithmetic: (9,999 − 167.15) / 24 = 409.66, so the mean becomes 576.81; the median moves only from 167.50 to 167.75. A mean that moves 42.65% or 245.08% while the median barely moves is not describing a typical price; it is describing the flag.

## 5. Pooled versus item-level averages

A single mean across all 96 prices is sometimes offered as “the price of a staple”. It is not: the four items sit at different levels, and pooling them weights each item equally regardless of consumption.

In [14]:
pooled = pd.Series({
    "pooled mean (96 prices)": df["price"].mean(),
    "pooled median (96 prices)": df["price"].median(),
})

print(pooled.round(3).to_string())
print()
print(df.groupby("item")["price"].agg(["mean", "median"]).round(3).to_string())

pooled mean (96 prices)      202.167
pooled median (96 prices)    192.250

                mean  median
item                        
Oil B        324.833  324.00
Pulse C      205.917  206.00
Staple A     111.167  111.75
Sweetener D  166.750  166.75


In [15]:
# The constant 6.0 city gap: a useful check, and not a 6.0% gap.
pair = df.pivot_table(index=["item", "date"], columns="city", values="price")
gap = (pair["City 2"] - pair["City 1"]).dropna()

print("complete pairs:", len(gap))
print("distinct gaps:", gap.round(6).unique())
print()
print(pd.DataFrame({
    "item mean": df.groupby("item")["price"].mean(),
    "6.0 as % of item mean": 6.0 / df.groupby("item")["price"].mean() * 100,
}).round(2).to_string())

complete pairs: 48
distinct gaps: [6.]

             item mean  6.0 as % of item mean
item                                         
Oil B           324.83                   1.85
Pulse C         205.92                   2.91
Staple A        111.17                   5.40
Sweetener D     166.75                   3.60


The gap is 6.0 PKR in every complete pair, but that is 5.40% of the Staple A mean and 1.85% of the Oil B mean. A sentence saying City 2 is “6% more expensive” would be wrong even though the 6.0 PKR gap is right. This is a useful example of a number that is correct in one unit and wrong in another.

## 6. Summary

In [16]:
summary = pd.DataFrame({
    "n": stats["n"],
    "mean": stats["mean"],
    "median": stats["median"],
    "first-to-last %": stats["pct_change_first_last"],
    "Jan->Mar monthly %": monthly["Jan->Mar %"],
}).round(2)
summary

,n,mean,median,first-to-last %,Jan->Mar monthly %
item,,,,,
Oil B,24,324.83,324.00,15.28,10.18
Pulse C,24,205.92,206.00,3.69,1.96
Staple A,24,111.17,111.75,12.98,10.69
Sweetener D,24,166.75,166.75,19.41,13.08


In [17]:
checks = {
    "96 rows": len(df) == 96,
    "4 items": df["item"].nunique() == 4,
    "2 cities": df["city"].nunique() == 2,
    "12 dates": df["date"].nunique() == 12,
    "no missing prices": int(df["price"].isna().sum()) == 0,
    "24 rows per item": bool(df.groupby("item").size().eq(24).all()),
    "12 rows per item-city": bool(df.groupby(["item", "city"]).size().eq(12).all()),
    "8 rows per date": bool(df.groupby("date").size().eq(8).all()),
    "48 city pairs": len(gap) == 48,
    "constant 6.0 gap": bool(gap.round(6).eq(6.0).all()),
    "Week 2 has 3 missing prices": int(w2["price"].isna().sum()) == 3,
}

for name, ok in checks.items():
    print(("PASS" if ok else "FAIL"), "-", name)

assert all(checks.values())

PASS - 96 rows
PASS - 4 items
PASS - 2 cities
PASS - 12 dates
PASS - no missing prices
PASS - 24 rows per item
PASS - 12 rows per item-city
PASS - 8 rows per date
PASS - 48 city pairs
PASS - constant 6.0 gap
PASS - Week 2 has 3 missing prices


## 7. Bottom line

- The Week 3 file is a complete, balanced panel: 96 prices, 4 items × 2 cities × 12 dates, no missing values, no duplicate keys.
- Every item is higher in the last week than in the first: +12.98% Staple A, +15.28% Oil B, +3.69% Pulse C, +19.41% Sweetener D.
- Mean and median agree within 0.83 PKR in this clean file, so the two neighbours are not disagreeing about the data. They are disagreeing about what “typical” means.
- In the Week 2 file, the flagged 1,250 raises Staple A’s mean by 42.65% but leaves the median unchanged; the flagged 9,999 raises Sweetener D’s mean by 245.08% while the median moves 0.15%. A mean on raw data can therefore say more than the data allow.
- A pooled mean of 202.17 is not the price of any item; the note reports item-level statistics.
- The 6.0 PKR city gap is not a 6% gap: it is 5.40% of the Staple A mean and 1.85% of the Oil B mean.
- The file is illustrative teaching data. No causal claim is supported.